## Azure AI Foundry Tokens and Usage
This notebook demonstrates how to understand and work with tokens when using Azure AI Foundry SDK.

## Overview
This hands-on session helps you understand Azure AI Foundry token usage and how it affects model interactions.

You can explore what tokens are and how they influence LLM model interaction: https://help.openai.com/en/articles/4936856-what-are-tokens-and-how-to-count-them

OpenAI Tokenizer: https://platform.openai.com/tokenizer or https://tiktokenizer.vercel.app/?model=gpt-4o

For further understanding about various GPT models and token limits: https://learn.microsoft.com/en-us/azure/ai-services/openai/concepts/models#gpt-4-and-gpt-4-turbo-models

#### Prerequisite
Please complete **00_Setup.ipynb** before running this notebook.

### 1. Install tiktoken python library

In [ ]:
# install tiktoken to count the number of tokens
%pip install tiktoken

#### Azure Authentication - see setup notebook for explanation. You may need to rerun this if the credetials expire

In [ ]:
# Azure Authentication using Helper Module
import os
from dotenv import load_dotenv
from azure_auth_helper import authenticate_azure

# Load environment variables
load_dotenv("./.env")

# Get tenant ID from environment variables
TENANT_ID = os.getenv('AZURE_TENANT_ID')
if not TENANT_ID:
    raise ValueError("AZURE_TENANT_ID not found in .env file. Please add it to your .env file.")

print(f"🏢 Using tenant ID: {TENANT_ID}")

# Authenticate with Azure using browser authentication
credential = authenticate_azure(
    auth_method='browser', 
    tenant_id=TENANT_ID
)

# Test that the credential actually works
print(f"🔍 Testing credential type: {type(credential).__name__}")
try:
    # Try to get a token to validate the credential
    token = credential.get_token("https://management.azure.com/.default")
    print("✅ Credential test successful!")
    print(f"Token expires: {token.expires_on}")
except Exception as e:
    print(f"❌ Credential test failed: {e}")
    print(f"   Error type: {type(e).__name__}")
    raise

print("🎉 Ready to use Azure AI Foundry!")

### 2. Import helper libraries and establish Foundry connection

In [ ]:
import os
import numpy as np
import tiktoken
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from datetime import datetime

# Load environment variables
load_dotenv("./.env")

project = AIProjectClient(
    endpoint=os.getenv("FOUNDRY_API_ENDPOINT"),
    credential=credential,
)

print("✅ Successfully connected to Azure AI Foundry")
print(f"📊 Project endpoint: {os.getenv('FOUNDRY_API_ENDPOINT')}")
print(f"🏢 Resource Group: {os.getenv('AZURE_RESOURCE_GROUP')}")
print(f"🔑 Using browser credential for project authentication")

### 3. Get deployed model

In [ ]:
# Get model deployment names from environment
gpt4o_model = os.getenv('GPT4O_DEPLOYMENT_NAME', 'gpt-4o')

# Now we can use the Foundry project client to get the OpenAI client
# This gives us all the Foundry benefits like monitoring and tracking
models = project.get_openai_client(api_version="2024-10-21")

print(f"🤖 Using model: {gpt4o_model}")
print("🎯 Ready for token analysis!")

### 4. Count the number of Tokens for given sentence

In [ ]:
# Sample prompt text to count the tokens
prompt = "Azure AI Foundry service is now available! Teams can start using Azure AI Foundry models now"

# Get encoding for the model
encoding = tiktoken.encoding_for_model(gpt4o_model)
print(f"Encoding: {encoding}")

# Encode the prompt to tokens
tokens = encoding.encode(prompt)
print(f'Total number of tokens: {len(tokens)}')
print(f'Tokens: {tokens}')
print(f'Words: {[encoding.decode([t]) for t in tokens]}')

### 5. Count tokens for a longer prompt

In [ ]:
# Create a longer prompt to demonstrate token usage
long_prompt = """NHS England, formerly the NHS Commissioning Board, is an executive non-departmental public body of the Department of Health and Social Care. It oversees the budget, planning, delivery and day-to-day operation of the commissioning side of the National Health Service in England as set out in the Health and Social Care Act 2012. It directly commissions NHS general practitioners, dentists, optometrists and some specialist services. The Secretary of State publishes annually a document known as the NHS mandate which specifies the objectives which the Board should seek to achieve. National Health Service (Mandate Requirements) Regulations are published each year to give legal force to the mandate. In 2018 it was announced that the organisation, while maintaining its statutory independence, would be merged with NHS Improvement, and seven 'single integrated regional teams' would be jointly established."""

# Analyze the token count
encoding = tiktoken.encoding_for_model(gpt4o_model)
tokens = encoding.encode(long_prompt)

print(f"📊 Prompt Analysis:")
print(f"Characters: {len(long_prompt)}")
print(f"Words: {len(long_prompt.split())}")
print(f"Tokens: {len(tokens)}")
print(f"First 10 tokens: {[encoding.decode([t]) for t in tokens[:10]]}")

In [ ]:
# Normal API call with the longer prompt
response = models.chat.completions.create(
    model=gpt4o_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": long_prompt + "\n\nPlease provide a brief summary of NHS England."}
    ],
    max_tokens=200
)

print(f"✅ Normal API call successful!")
print(f"Prompt tokens: {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"\nResponse: {response.choices[0].message.content}")

### 6. Testing max_tokens Parameter

The `max_tokens` parameter controls the maximum length of the model's response. Let's test what happens when we set a very low limit:

In [ ]:
# Test max_tokens limitation
response = models.chat.completions.create(
    model=gpt4o_model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Please provide a detailed summary."},
        {"role": "user", "content": long_prompt + "\n\nPlease provide a comprehensive summary of NHS England, including all key points, history, and organizational structure."}
    ],
    max_tokens=30  # Very small limit to force truncation
)

print(f"🔬 max_tokens Test Results:")
print(f"Max tokens requested: 30")
print(f"Completion tokens used: {response.usage.completion_tokens}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"\nTruncated response: {response.choices[0].message.content}")

if response.choices[0].finish_reason == "length":
    print("\n💡 Response was cut off due to max_tokens limit!")

### 7. Testing Context Window Limits

⚠️ **This does use a significant number of tokens. Skip if costings are a concern.**

Each model has a maximum context window (total tokens for input + output). GPT-4o has approximately 128,000 tokens. Let's test what happens when we exceed this limit. You may also see a 429 error if you are rate limited on tokens.

In [ ]:
# Create an extremely large prompt to exceed context window
extremely_large_prompt = long_prompt * 1500  # Repeat text many times
large_tokens = encoding.encode(extremely_large_prompt)

print(f"📏 Context Window Test Setup:")
print(f"Large prompt tokens: {len(large_tokens):,}")
print(f"GPT-4o context limit: ~128,000 tokens")
print(f"Exceeds limit: {'Yes' if len(large_tokens) > 128000 else 'No'}")
print()

In [ ]:
# Try to send the extremely large prompt - this should fail
try:
    response = models.chat.completions.create(
        model=gpt4o_model,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": extremely_large_prompt + "\n\nPlease summarize."}
        ],
        max_tokens=100
    )
    
    print("❌ Unexpected success - was your prompt long enough?")
    
except Exception as e:
    print(f"✅ Expected error occurred:")
    print(f"Error type: {type(e).__name__}")
    
    if "maximum context length" in str(e).lower() or "context_length_exceeded" in str(e).lower():
        print("💡 Context window limit exceeded!")
        print(f"Attempted tokens: {len(large_tokens):,}")
        print(f"Limit: ~128,000 tokens")
        print(f"Exceeded by: {len(large_tokens) - 128000:,} tokens")
    else:
        print(f"Different error: {e}")

#### Key Learnings

**Normal Usage:**
- Monitor token consumption for cost optimization
- Typical prompts use hundreds to thousands of tokens

**Output Limits (`max_tokens`):**
- Controls maximum response length only
- When hit, `finish_reason` becomes "length" 
- Response gets truncated mid-sentence

**Input Limits (Context Window):**
- Each model has a maximum total token limit
- Includes both input prompt AND expected output space
- Exceeding causes immediate API error

**Best Practices:**
- Always handle token errors gracefully
- Use chunking strategies for large documents
- Consider token costs when designing prompts